In [1]:
import pandas as pd

regex = regular expression also r" " here r is representing as raw text/string and \s for whitespaces (like spaces, tabs, newline) and + is for one or more.

In [3]:
with open("dreaddit-train.csv", "r", encoding="utf-8") as f:
    texts = [line.strip() for line in f if line.strip()]

df = pd.DataFrame({
    "raw_text": texts
})

In [4]:
df

,raw_text
0,"subreddit,post_id,sentence_range,text,id,label..."
1,"ptsd,8601tu,""(15, 20)"",""He said he had not fel..."
2,"assistance,8lbrx9,""(0, 5)"",""Hey there r/assist..."
3,"ptsd,9ch1zh,""(15, 20)"",""My mom then hit me wit..."
4,"relationships,7rorpp,""[5, 10]"",""until i met my..."
...,...
2834,"relationships,7oee1t,""[35, 40]"",""* Her, a week..."
2835,"ptsd,9p4ung,""[20, 25]"",""I don't have the abili..."
2836,"anxiety,9nam6l,""(5, 10)"",""In case this is the ..."
2837,"almosthomeless,5y53ya,""[5, 10]"",Do you find th..."


In [ ]:
df.shape

(2839, 1)

In [ ]:
import html

df["clean_text"] = df["raw_text"]

In [ ]:
# HTML entities
df["clean_text"] = df["clean_text"].apply(html.unescape)

In [ ]:
# Invisible Unicode characters
df["clean_text"] = df["clean_text"].str.replace(
    r"[\u200B\u200C\u200D\uFEFF]",
    "",
    regex=True
)

In [ ]:
# URLs
df["clean_text"] = df["clean_text"].str.replace(
    r"https?://\S+|www\.\S+",
    " URL ",
    regex=True
)

In [ ]:
# Mentions
df["clean_text"] = df["clean_text"].str.replace(
    r"@\w+",
    " USER ",
    regex=True
)

In [ ]:
# Whitespace
df["clean_text"] = (
    df["clean_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
# Lowercase
df["clean_text"] = df["clean_text"].str.lower()

In [ ]:
df = df.drop_duplicates(
    subset="raw_text"
).reset_index(drop=True)

**Data quality checking**

In [ ]:
print("Missing values:")
df.isnull().sum()

Missing values:


,0
raw_text,0
clean_text,0


In [ ]:
# Empty texts
empty_count = (df["clean_text"].str.strip() == "").sum()
print("\nEmpty texts:", empty_count)


Empty texts: 0


In [ ]:
# Text length
df["text_length"] = df["clean_text"].str.len()

In [ ]:
print("\nText length statistics:")
print(df["text_length"].describe())


Text length statistics:
count    2839.000000
mean      954.785488
std       186.747417
min       379.000000
25%       840.000000
50%       924.000000
75%      1049.000000
max      2206.000000
Name: text_length, dtype: float64


In [ ]:
display(
    df[["raw_text", "clean_text"]]
    .sample(20, random_state=42)
)

,raw_text,clean_text
415,"assistance,80y9u6,""[0, 5]"",""My phone is a prep...","assistance,80y9u6,""[0, 5]"",""my phone is a prep..."
1302,"anxiety,9n13q1,""[0, 5]"",""Hey, r/Anxiety! &#x20...","anxiety,9n13q1,""[0, 5]"",""hey, r/anxiety! i'm a..."
2475,"relationships,7nnb5k,""(0, 5)"",""I [26F] have be...","relationships,7nnb5k,""(0, 5)"",""i [26f] have be..."
1340,"anxiety,8c5bdu,""[0, 5]"",""During my break at wo...","anxiety,8c5bdu,""[0, 5]"",""during my break at wo..."
1599,"survivorsofabuse,9co0fc,""[0, 5]"",""This post wa...","survivorsofabuse,9co0fc,""[0, 5]"",""this post wa..."
354,"homeless,6d9hkr,""[0, 5]"",""<url> My mother is 6...","homeless,6d9hkr,""[0, 5]"",""<url> my mother is 6..."
965,"anxiety,5nubd7,""[5, 10]"",I could easily see my...","anxiety,5nubd7,""[5, 10]"",i could easily see my..."
2067,"ptsd,9nc98u,""[25, 30]"",And I don't even rememb...","ptsd,9nc98u,""[25, 30]"",and i don't even rememb..."
527,"domesticviolence,78iim7,""(10, 15)"",""Not becaus...","domesticviolence,78iim7,""(10, 15)"",""not becaus..."
2418,"assistance,9chhpj,""(2, 7)"",""I can't afford one...","assistance,9chhpj,""(2, 7)"",""i can't afford one..."


In [ ]:
display(
    df[df["text_length"] < 20][
        ["raw_text", "clean_text", "text_length"]
    ].head(20)
)

,raw_text,clean_text,text_length


## Load Dreaddit Labeled Data

In [ ]:
train_original = pd.read_csv('dreaddit-train (1).csv')
test_original = pd.read_csv('dreaddit-test.csv')

print("Train shape:", train_original.shape)
print("Test shape:", test_original.shape)

Train shape: (2838, 116)
Test shape: (715, 116)


In [ ]:
import re, html

def light_clean(t):
    t = html.unescape(str(t))
    t = re.sub(r"[\u200B\u200C\u200D\uFEFF]", "", t)
    t = re.sub(r"https?://\S+|www\.\S+", " URL ", t)
    t = re.sub(r"@\w+|\bu/\w+", " USER ", t)
    return re.sub(r"\s+", " ", t).strip()

train = train_original.copy()
test  = test_original.copy()
for d in (train, test):
    d["bert_text"]  = d["text"].apply(light_clean)
    d["clean_text"] = d["bert_text"].str.lower()

train = train[train["bert_text"].str.len() >= 20].drop_duplicates(subset="clean_text")
test  = test[test["bert_text"].str.len() >= 20].drop_duplicates(subset="clean_text")
test  = test[~test["clean_text"].isin(train["clean_text"])]
train = train.reset_index(drop=True)
test  = test.reset_index(drop=True)

print("Train:", train.shape, "| Test:", test.shape)
print("Shared posts:", len(set(train["post_id"]) & set(test["post_id"])))

Train: (2819, 118) | Test: (712, 118)
Shared posts: 0


In [ ]:
print("Train columns:")
print(train_original.columns.tolist())

print("\nTest columns:")
print(test_original.columns.tolist())

display(train_original.head())
display(test_original.head())

Train columns:
['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female', 'lex_liwc_male', 'lex_liwc_cogproc', 'lex_liwc_insight', 'lex_liwc_cause', 'lex_liwc_discrep', 'lex_liwc_tentat', 'lex_liwc_certain', 'lex_liwc_diff

,subreddit,post_id,sentence_range,text,id,label,confidence,social_timestamp,social_karma,syntax_ari,...,lex_dal_min_pleasantness,lex_dal_min_activation,lex_dal_min_imagery,lex_dal_avg_activation,lex_dal_avg_imagery,lex_dal_avg_pleasantness,social_upvote_ratio,social_num_comments,syntax_fk_grade,sentiment
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",33181,1,0.8,1521614353,5,1.806818,...,1.000,1.1250,1.0,1.77000,1.52211,1.89556,0.86,1,3.253573,-0.002742
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",2606,0,1.0,1527009817,4,9.429737,...,1.125,1.0000,1.0,1.69586,1.62045,1.88919,0.65,2,8.828316,0.292857
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,38816,1,0.8,1535935605,2,7.769821,...,1.000,1.1429,1.0,1.83088,1.58108,1.85828,0.67,0,7.841667,0.011894
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",239,1,0.6,1516429555,0,2.667798,...,1.000,1.1250,1.0,1.75356,1.52114,1.98848,0.50,5,4.104027,0.141671
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1421,1,0.8,1539809005,24,7.554238,...,1.000,1.1250,1.0,1.77644,1.64872,1.81456,1.00,1,7.910952,-0.204167


,id,subreddit,post_id,sentence_range,text,label,confidence,social_timestamp,social_karma,syntax_ari,...,lex_dal_min_pleasantness,lex_dal_min_activation,lex_dal_min_imagery,lex_dal_avg_activation,lex_dal_avg_imagery,lex_dal_avg_pleasantness,social_upvote_ratio,social_num_comments,syntax_fk_grade,sentiment
0,896,relationships,7nu7as,"[50, 55]","Its like that, if you want or not.“ ME: I have...",0,0.8,1.514981e+09,22,-1.238793,...,1.0000,1.2000,1.0,1.65864,1.32245,1.80264,0.63,62,-0.148707,0.000000
1,19059,anxiety,680i6d,"(5, 10)",I man the front desk and my title is HR Custom...,0,1.0,1.493348e+09,5,7.684583,...,1.4000,1.1250,1.0,1.69133,1.69180,1.97249,1.00,2,7.398222,-0.065909
2,7977,ptsd,8eeu1t,"(5, 10)",We'd be saving so much money with this new hou...,1,1.0,1.524517e+09,10,2.360408,...,1.1429,1.0000,1.0,1.70974,1.52985,1.86108,1.00,8,3.149288,-0.036818
3,1214,ptsd,8d28vu,"[2, 7]","My ex used to shoot back with ""Do you want me ...",1,0.5,1.524018e+09,5,5.997000,...,1.0000,1.3000,1.0,1.72615,1.52000,1.84909,1.00,7,6.606000,-0.066667
4,1965,relationships,7r1e85,"[23, 28]",I haven’t said anything to him yet because I’m...,0,0.8,1.516200e+09,138,4.649418,...,1.1250,1.1429,1.0,1.75642,1.43582,1.91725,0.84,70,4.801869,0.141667


In [ ]:
print(train_original["label"].value_counts(dropna=False))
print(train_original["label"].unique())

label
1    1488
0    1350
Name: count, dtype: int64
[1 0]


In [ ]:
dreaddit_labeled = pd.concat(
    [train_original, test_original],
    ignore_index=True
)

print(dreaddit_labeled.shape)

(3553, 116)


In [ ]:
display(dreaddit_labeled.head())
print(dreaddit_labeled.columns.tolist())

,subreddit,post_id,sentence_range,text,id,label,confidence,social_timestamp,social_karma,syntax_ari,...,lex_dal_min_pleasantness,lex_dal_min_activation,lex_dal_min_imagery,lex_dal_avg_activation,lex_dal_avg_imagery,lex_dal_avg_pleasantness,social_upvote_ratio,social_num_comments,syntax_fk_grade,sentiment
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",33181,1,0.8,1.521614e+09,5,1.806818,...,1.000,1.1250,1.0,1.77000,1.52211,1.89556,0.86,1,3.253573,-0.002742
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",2606,0,1.0,1.527010e+09,4,9.429737,...,1.125,1.0000,1.0,1.69586,1.62045,1.88919,0.65,2,8.828316,0.292857
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,38816,1,0.8,1.535936e+09,2,7.769821,...,1.000,1.1429,1.0,1.83088,1.58108,1.85828,0.67,0,7.841667,0.011894
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",239,1,0.6,1.516430e+09,0,2.667798,...,1.000,1.1250,1.0,1.75356,1.52114,1.98848,0.50,5,4.104027,0.141671
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1421,1,0.8,1.539809e+09,24,7.554238,...,1.000,1.1250,1.0,1.77644,1.64872,1.81456,1.00,1,7.910952,-0.204167


['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female', 'lex_liwc_male', 'lex_liwc_cogproc', 'lex_liwc_insight', 'lex_liwc_cause', 'lex_liwc_discrep', 'lex_liwc_tentat', 'lex_liwc_certain', 'lex_liwc_differ', 'lex_liwc_

In [ ]:
print(dreaddit_labeled.columns.tolist())
print(dreaddit_labeled.shape)
display(dreaddit_labeled.head())

['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female', 'lex_liwc_male', 'lex_liwc_cogproc', 'lex_liwc_insight', 'lex_liwc_cause', 'lex_liwc_discrep', 'lex_liwc_tentat', 'lex_liwc_certain', 'lex_liwc_differ', 'lex_liwc_

,subreddit,post_id,sentence_range,text,id,label,confidence,social_timestamp,social_karma,syntax_ari,...,lex_dal_min_pleasantness,lex_dal_min_activation,lex_dal_min_imagery,lex_dal_avg_activation,lex_dal_avg_imagery,lex_dal_avg_pleasantness,social_upvote_ratio,social_num_comments,syntax_fk_grade,sentiment
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",33181,1,0.8,1.521614e+09,5,1.806818,...,1.000,1.1250,1.0,1.77000,1.52211,1.89556,0.86,1,3.253573,-0.002742
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",2606,0,1.0,1.527010e+09,4,9.429737,...,1.125,1.0000,1.0,1.69586,1.62045,1.88919,0.65,2,8.828316,0.292857
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,38816,1,0.8,1.535936e+09,2,7.769821,...,1.000,1.1429,1.0,1.83088,1.58108,1.85828,0.67,0,7.841667,0.011894
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",239,1,0.6,1.516430e+09,0,2.667798,...,1.000,1.1250,1.0,1.75356,1.52114,1.98848,0.50,5,4.104027,0.141671
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1421,1,0.8,1.539809e+09,24,7.554238,...,1.000,1.1250,1.0,1.77644,1.64872,1.81456,1.00,1,7.910952,-0.204167


In [ ]:
print(dreaddit_labeled["label"].value_counts(dropna=False))

label
1    1857
0    1696
Name: count, dtype: int64


In [ ]:
print(dreaddit_labeled.columns.tolist())

['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female', 'lex_liwc_male', 'lex_liwc_cogproc', 'lex_liwc_insight', 'lex_liwc_cause', 'lex_liwc_discrep', 'lex_liwc_tentat', 'lex_liwc_certain', 'lex_liwc_differ', 'lex_liwc_

In [ ]:
display(
    dreaddit_labeled.head(10)
)

,subreddit,post_id,sentence_range,text,id,label,confidence,social_timestamp,social_karma,syntax_ari,...,lex_dal_min_pleasantness,lex_dal_min_activation,lex_dal_min_imagery,lex_dal_avg_activation,lex_dal_avg_imagery,lex_dal_avg_pleasantness,social_upvote_ratio,social_num_comments,syntax_fk_grade,sentiment
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",33181,1,0.8,1.521614e+09,5,1.806818,...,1.000,1.1250,1.0,1.77000,1.52211,1.89556,0.86,1,3.253573,-0.002742
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",2606,0,1.0,1.527010e+09,4,9.429737,...,1.125,1.0000,1.0,1.69586,1.62045,1.88919,0.65,2,8.828316,0.292857
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,38816,1,0.8,1.535936e+09,2,7.769821,...,1.000,1.1429,1.0,1.83088,1.58108,1.85828,0.67,0,7.841667,0.011894
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",239,1,0.6,1.516430e+09,0,2.667798,...,1.000,1.1250,1.0,1.75356,1.52114,1.98848,0.50,5,4.104027,0.141671
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1421,1,0.8,1.539809e+09,24,7.554238,...,1.000,1.1250,1.0,1.77644,1.64872,1.81456,1.00,1,7.910952,-0.204167
5,relationships,7tx7et,"(30, 35)",I think he doesn't want to put in the effort f...,17554,1,1.0,1.517274e+09,2,3.531124,...,1.000,1.1818,1.0,1.70415,1.44396,1.82046,0.75,5,4.338429,-0.550000
6,domesticviolence,7iphly,"[25, 30]",It was a big company so luckily I didn't have ...,165,0,0.8,1.512854e+09,6,8.331463,...,1.000,1.1250,1.0,1.66985,1.57757,1.89926,0.88,29,8.394244,0.277381
7,anxiety,5m3k80,"(5, 10)",It cleared up and I was okay but. On Monday ...,33053,1,0.8,1.483582e+09,1,0.403679,...,1.000,1.1429,1.0,1.69549,1.44860,1.88963,1.00,0,1.028654,-0.079630
8,relationships,7nhy1v,"(50, 55)",I actually give an assistant half my emergency...,7581,1,0.6,1.514844e+09,134,5.381111,...,1.000,1.0000,1.0,1.73571,1.71471,1.93607,0.94,43,5.991049,-0.272917
9,assistance,61eiq6,"[15, 20]",I just feel like the street life has fucked my...,1517,1,1.0,1.490428e+09,20,1.562351,...,1.000,1.1250,1.0,1.74956,1.67353,1.89320,0.80,6,2.851935,-0.117262


In [ ]:
print(dreaddit_labeled.dtypes)

subreddit                    object
post_id                      object
sentence_range               object
text                         object
id                            int64
                             ...   
lex_dal_avg_pleasantness    float64
social_upvote_ratio         float64
social_num_comments           int64
syntax_fk_grade             float64
sentiment                   float64
Length: 116, dtype: object


In [ ]:
print(
    "Duplicate labeled texts:",
    dreaddit_labeled["text"].duplicated().sum()
)

Duplicate labeled texts: 21


In [ ]:
print(dreaddit_labeled.columns.tolist())
display(dreaddit_labeled.head())
print(dreaddit_labeled.dtypes)
print(dreaddit_labeled.isnull().sum())

['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female', 'lex_liwc_male', 'lex_liwc_cogproc', 'lex_liwc_insight', 'lex_liwc_cause', 'lex_liwc_discrep', 'lex_liwc_tentat', 'lex_liwc_certain', 'lex_liwc_differ', 'lex_liwc_

,subreddit,post_id,sentence_range,text,id,label,confidence,social_timestamp,social_karma,syntax_ari,...,lex_dal_min_pleasantness,lex_dal_min_activation,lex_dal_min_imagery,lex_dal_avg_activation,lex_dal_avg_imagery,lex_dal_avg_pleasantness,social_upvote_ratio,social_num_comments,syntax_fk_grade,sentiment
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",33181,1,0.8,1.521614e+09,5,1.806818,...,1.000,1.1250,1.0,1.77000,1.52211,1.89556,0.86,1,3.253573,-0.002742
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",2606,0,1.0,1.527010e+09,4,9.429737,...,1.125,1.0000,1.0,1.69586,1.62045,1.88919,0.65,2,8.828316,0.292857
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,38816,1,0.8,1.535936e+09,2,7.769821,...,1.000,1.1429,1.0,1.83088,1.58108,1.85828,0.67,0,7.841667,0.011894
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",239,1,0.6,1.516430e+09,0,2.667798,...,1.000,1.1250,1.0,1.75356,1.52114,1.98848,0.50,5,4.104027,0.141671
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1421,1,0.8,1.539809e+09,24,7.554238,...,1.000,1.1250,1.0,1.77644,1.64872,1.81456,1.00,1,7.910952,-0.204167


subreddit                    object
post_id                      object
sentence_range               object
text                         object
id                            int64
                             ...   
lex_dal_avg_pleasantness    float64
social_upvote_ratio         float64
social_num_comments           int64
syntax_fk_grade             float64
sentiment                   float64
Length: 116, dtype: object
subreddit                   0
post_id                     0
sentence_range              0
text                        0
id                          0
                           ..
lex_dal_avg_pleasantness    0
social_upvote_ratio         0
social_num_comments         0
syntax_fk_grade             0
sentiment                   0
Length: 116, dtype: int64


## Understanding Dreaddit Labels

In [ ]:
display(
    dreaddit_labeled[
        ["subreddit", "post_id", "sentence_range", "text", "label", "confidence"]
    ].head(10)
)

,subreddit,post_id,sentence_range,text,label,confidence
0,ptsd,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",1,0.8
1,assistance,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",0,1.0
2,ptsd,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,1,0.8
3,relationships,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",1,0.6
4,survivorsofabuse,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1,0.8
5,relationships,7tx7et,"(30, 35)",I think he doesn't want to put in the effort f...,1,1.0
6,domesticviolence,7iphly,"[25, 30]",It was a big company so luckily I didn't have ...,0,0.8
7,anxiety,5m3k80,"(5, 10)",It cleared up and I was okay but. On Monday ...,1,0.8
8,relationships,7nhy1v,"(50, 55)",I actually give an assistant half my emergency...,1,0.6
9,assistance,61eiq6,"[15, 20]",I just feel like the street life has fucked my...,1,1.0


In [ ]:
display(
    df[["raw_text", "clean_text"]].head(10)
)

,raw_text,clean_text
0,"subreddit,post_id,sentence_range,text,id,label...","subreddit,post_id,sentence_range,text,id,label..."
1,"ptsd,8601tu,""(15, 20)"",""He said he had not fel...","ptsd,8601tu,""(15, 20)"",""he said he had not fel..."
2,"assistance,8lbrx9,""(0, 5)"",""Hey there r/assist...","assistance,8lbrx9,""(0, 5)"",""hey there r/assist..."
3,"ptsd,9ch1zh,""(15, 20)"",""My mom then hit me wit...","ptsd,9ch1zh,""(15, 20)"",""my mom then hit me wit..."
4,"relationships,7rorpp,""[5, 10]"",""until i met my...","relationships,7rorpp,""[5, 10]"",""until i met my..."
5,"survivorsofabuse,9p2gbc,""[0, 5]"",""October is D...","survivorsofabuse,9p2gbc,""[0, 5]"",""october is d..."
6,"relationships,7tx7et,""(30, 35)"",""I think he do...","relationships,7tx7et,""(30, 35)"",""i think he do..."
7,"domesticviolence,7iphly,""[25, 30]"",""It was a b...","domesticviolence,7iphly,""[25, 30]"",""it was a b..."
8,"anxiety,5m3k80,""(5, 10)"", It cleared up and I...","anxiety,5m3k80,""(5, 10)"", it cleared up and i ..."
9,"relationships,7nhy1v,""(50, 55)"",""I actually gi...","relationships,7nhy1v,""(50, 55)"",""i actually gi..."


In [ ]:
display(
    dreaddit_labeled[
        ["post_id", "sentence_range", "text", "label", "confidence"]
    ].head(10)
)

,post_id,sentence_range,text,label,confidence
0,8601tu,"(15, 20)","He said he had not felt that way before, sugge...",1,0.8
1,8lbrx9,"(0, 5)","Hey there r/assistance, Not sure if this is th...",0,1.0
2,9ch1zh,"(15, 20)",My mom then hit me with the newspaper and it s...,1,0.8
3,7rorpp,"[5, 10]","until i met my new boyfriend, he is amazing, h...",1,0.6
4,9p2gbc,"[0, 5]",October is Domestic Violence Awareness Month a...,1,0.8
5,7tx7et,"(30, 35)",I think he doesn't want to put in the effort f...,1,1.0
6,7iphly,"[25, 30]",It was a big company so luckily I didn't have ...,0,0.8
7,5m3k80,"(5, 10)",It cleared up and I was okay but. On Monday ...,1,0.8
8,7nhy1v,"(50, 55)",I actually give an assistant half my emergency...,1,0.6
9,61eiq6,"[15, 20]",I just feel like the street life has fucked my...,1,1.0


In [ ]:
print("Your dataset:")
print(df["raw_text"].str.len().describe())

print("\nDreaddit labeled:")
print(dreaddit_labeled["text"].str.len().describe())

Your dataset:
count    2839.000000
mean      955.138781
std       186.667795
min       379.000000
25%       840.000000
50%       925.000000
75%      1049.000000
max      2206.000000
Name: raw_text, dtype: float64

Dreaddit labeled:
count    3553.000000
mean      447.669856
std       167.949952
min         6.000000
25%       338.000000
50%       421.000000
75%       527.000000
max      1639.000000
Name: text, dtype: float64


In [ ]:
import re, html
import pandas as pd

def clean(t):
    t = html.unescape(t)
    t = re.sub(r"[\u200B\u200C\u200D\uFEFF]", "", t)
    t = re.sub(r"https?://\S+|www\.\S+", " URL ", t)
    t = re.sub(r"@\w+", " USER ", t)
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t

data = pd.concat([train_original, test_original], ignore_index=True)
data["clean_text"] = data["text"].apply(clean)
data = data.drop_duplicates(subset="clean_text").reset_index(drop=True)
print(data.shape, data["label"].value_counts().to_dict())

(3532, 117) {1: 1849, 0: 1683}


In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. numeric columns already in your CSV (LIWC word-category counts etc.)
num_cols = [c for c in train.columns
            if c.startswith(("lex_", "syntax_", "social_")) and c != "social_timestamp"
            and pd.api.types.is_numeric_dtype(train[c])]

# 2. fit the text and number transformers on TRAIN only
vec_word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True).fit(train["clean_text"])
vec_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True).fit(train["clean_text"])
scaler = StandardScaler().fit(train[num_cols])

def make_X(d):
    return hstack([
        vec_word.transform(d["clean_text"]),
        vec_char.transform(d["clean_text"]),
        csr_matrix(scaler.transform(d[num_cols])) * 0.3,
    ]).tocsr()

X_train, X_test = make_X(train), make_X(test)

# 3. train + test
clf = LogisticRegression(max_iter=5000, C=2)
clf.fit(X_train, train["label"])
pred_classic = clf.predict(X_test)
prob_classic = clf.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(test["label"], pred_classic), 4))
print("F1      :", round(f1_score(test["label"], pred_classic), 4))
print(classification_report(test["label"], pred_classic, target_names=["no stress", "stress"]))

Accuracy: 0.7654
F1      : 0.7817
              precision    recall  f1-score   support

   no stress       0.78      0.71      0.75       345
      stress       0.75      0.81      0.78       367

    accuracy                           0.77       712
   macro avg       0.77      0.76      0.76       712
weighted avg       0.77      0.77      0.76       712



In [ ]:
!pip install -q transformers datasets accelerate

import numpy as np, torch
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

set_seed(42)
MODEL_NAME = "roberta-base"
print("GPU:", torch.cuda.is_available())        # must say True

# validation split from train (by post_id, no leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
a, b = next(gss.split(train, train["label"], groups=train["post_id"]))
tr, va = train.iloc[a], train.iloc[b]

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(d):
    ds = Dataset.from_dict({"text": d["bert_text"].tolist(), "label": d["label"].tolist()})
    return ds.map(lambda x: tok(x["text"], truncation=True, max_length=128), batched=True)

ds_tr, ds_va, ds_te = make_ds(tr), make_ds(va), make_ds(test)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"acc": accuracy_score(p.label_ids, preds), "f1": f1_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir="stress_model", num_train_epochs=3, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1",
    weight_decay=0.01, fp16=torch.cuda.is_available(), seed=42, report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

out = trainer.predict(ds_te)
prob_roberta = torch.softmax(torch.tensor(out.predictions), dim=1).numpy()[:, 1]
pred_roberta = (prob_roberta > 0.5).astype(int)

print("Accuracy:", round(accuracy_score(test["label"], pred_roberta), 4))
print("F1      :", round(f1_score(test["label"], pred_roberta), 4))
print(classification_report(test["label"], pred_roberta, target_names=["no stress", "stress"]))

trainer.save_model("stress_model_final")
tok.save_pretrained("stress_model_final")

GPU: True


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (10 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Accuracy: 0.7865
F1      : 0.8114
              precision    recall  f1-score   support

   no stress       0.85      0.68      0.75       345
      stress       0.74      0.89      0.81       367

    accuracy                           0.79       712
   macro avg       0.80      0.78      0.78       712
weighted avg       0.80      0.79      0.78       712



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('stress_model_final/tokenizer_config.json',
 'stress_model_final/tokenizer.json')

In [ ]:
prob_mix = (prob_classic + prob_roberta) / 2
pred_mix = (prob_mix > 0.5).astype(int)
print("Ensemble accuracy:", round(accuracy_score(test["label"], pred_mix), 4))
print("Ensemble F1      :", round(f1_score(test["label"], pred_mix), 4))

Ensemble accuracy: 0.7851
Ensemble F1      : 0.8066


In [48]:
test_eval = test.copy()
test_eval["pred"] = pred_roberta
test_eval["p_stress"] = prob_roberta

wrong = test_eval[test_eval["label"] != test_eval["pred"]]
print("Wrong:", len(wrong), "of", len(test_eval))

for _, r in wrong.sample(10, random_state=1).iterrows():
    print(f"TRUE={r['label']} PRED={r['pred']} P(stress)={r['p_stress']:.2f} [{r['subreddit']}]")
    print(r["bert_text"][:300], "\n")

test_eval["correct"] = test_eval["label"] == test_eval["pred"]
test_eval.groupby("subreddit")["correct"].agg(["mean", "count"]).round(3).sort_values("mean")

Wrong: 152 of 712
TRUE=0 PRED=1 P(stress)=0.92 [survivorsofabuse]
I don't know how to maintain my relationship with my brother while he still lives under my parents' control \(which covers everything from finances, to communication with other people, to leaving their house \-\- or rather not being outside the house without them other than for school and work\). I' 

TRUE=1 PRED=0 P(stress)=0.35 [ptsd]
Yeah, not so much. I had a minor surgery on my neck today that just used local anesthetic, so I was fully conscious for it. Everything was fine since I'm not squeamish - until I realized they needed to strap me down to make sure I didn't move. It's a delicate procedure; I understood. But I hadn't be 

TRUE=1 PRED=0 P(stress)=0.26 [relationships]
Almost too well. She extends our date twice, buys us a drink each, starts a make out session which lasts for 2 hours, calls me hot, and although neither of us had a house free she gave me oral too. Although she said she had a great time - which I 

,mean,count
subreddit,,
food_pantry,0.667,6
relationships,0.711,142
survivorsofabuse,0.716,67
homeless,0.750,52
almosthomeless,0.789,19
ptsd,0.803,127
anxiety,0.816,147
domesticviolence,0.847,72
stress,0.857,14
